<a href="https://colab.research.google.com/github/gitly-br/PSA_pred_models/blob/modelos_v2/notebooks/Processamento_chamados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introdução

# Preparação

## Imports

In [23]:
import pandas as pd
import os
import copy
import json

## Funções

Leitura de arquivos. Primeiro tenta achar localmente, depois busca pelo link do drive.

In [24]:
def load_csv_dataframe(file_name, drive_link=None):
    local_path = file_name
    if os.path.exists(local_path):
        print(f"'{file_name}' found locally. Loading...")
        df = pd.read_csv(local_path)
        return df.copy()
    elif drive_link:
        print(f"'{file_name}' not found locally. Attempting to load from Google Drive link...")
        try:
            # Attempt to convert a standard Google Drive share link to a direct download link
            if "drive.google.com" in drive_link and "view" in drive_link:
                file_id = drive_link.split('/d/')[-1].split('/view')[0].split('?')[0]
                direct_download_link = f"https://drive.google.com/uc?export=download&id={file_id}"
                print(f"Converted Google Drive link to direct download: {direct_download_link}")
                df = pd.read_csv(direct_download_link)
            else:
                df = pd.read_csv(drive_link) # Assume it's already a direct readable URL

            print(f"Successfully loaded from Google Drive link.")
            return copy.deepcopy(df)
        except Exception as e:
            print(f"Error loading from Google Drive link: {e}")
            raise FileNotFoundError(
                f"'{file_name}' not found locally and could not be loaded from Google Drive link."
            )
    else:
        raise FileNotFoundError(
            f"'{file_name}' not found locally and no Google Drive link provided."
        )

def load_json_to_dict(file_name, drive_link=None):
    local_path = file_name
    if os.path.exists(local_path):
        print(f"'{file_name}' found locally. Loading...")
        with open(local_path, 'r') as f:
            data = json.load(f)
        return copy.deepcopy(data)
    elif drive_link:
        print(f"'{file_name}' not found locally. Attempting to load from Google Drive link...")
        try:
            import requests
            # Attempt to convert a standard Google Drive share link to a direct download link
            if "drive.google.com" in drive_link and "view" in drive_link:
                file_id = drive_link.split('/d/')[-1].split('/view')[0].split('?')[0]
                direct_download_link = f"https://drive.google.com/uc?export=download&id={file_id}"
                print(f"Converted Google Drive link to direct download: {direct_download_link}")
                response = requests.get(direct_download_link)
            else:
                response = requests.get(drive_link)

            response.raise_for_status() # Raise an exception for HTTP errors
            data = json.loads(response.text)
            print(f"Successfully loaded from Google Drive link.")
            return copy.deepcopy(data)
        except requests.exceptions.RequestException as e:
            print(f"Error fetching from Google Drive link: {e}")
            raise FileNotFoundError(
                f"'{file_name}' not found locally and could not be loaded from Google Drive link (Request Error)."
            )
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON from Google Drive link: {e}")
            raise ValueError(
                f"Could not decode JSON from Google Drive link for '{file_name}'."
            )
        except Exception as e:
            print(f"An unexpected error occurred loading from Google Drive link: {e}")
            raise FileNotFoundError(
                f"'{file_name}' not found locally and could not be loaded from Google Drive link."
            )
    else:
        raise FileNotFoundError(
            f"'{file_name}' not found locally and no Google Drive link provided."
        )


## Datasets externos

Chamados sem processamento algum

In [64]:
df_chamados_raw = load_csv_dataframe(
    "/content/chamados_raw.csv",
    drive_link="https://drive.google.com/file/d/1jzFVDT9LYGF7gk3e68FGOVF1r6k60wMJ/view?usp=drive_link"
    )

'/content/chamados_raw.csv' not found locally. Attempting to load from Google Drive link...
Converted Google Drive link to direct download: https://drive.google.com/uc?export=download&id=1jzFVDT9LYGF7gk3e68FGOVF1r6k60wMJ
Successfully loaded from Google Drive link.


Definição de bairros por bacia

In [72]:
bacias_bairros = load_json_to_dict(
    "/content/bacias.json",
    drive_link="https://drive.google.com/file/d/15oplqaHWY20ad7aL8Dt7AMvDFe_uTtq9/view?usp=drive_link"
)

'/content/bacias.json' not found locally. Attempting to load from Google Drive link...
Converted Google Drive link to direct download: https://drive.google.com/uc?export=download&id=15oplqaHWY20ad7aL8Dt7AMvDFe_uTtq9
Successfully loaded from Google Drive link.


Alagamentos confirmados por notícias

In [95]:
df_alagamentos_confirmados = load_csv_dataframe(
    "/content/chamados_raw.csv",
    drive_link="https://drive.google.com/file/d/126R0l_TKkkr5-tFkgWXPaXs9mktjYecG/view?usp=drive_link"
)

'/content/chamados_raw.csv' not found locally. Attempting to load from Google Drive link...
Converted Google Drive link to direct download: https://drive.google.com/uc?export=download&id=126R0l_TKkkr5-tFkgWXPaXs9mktjYecG
Successfully loaded from Google Drive link.


# Separação por bairros

Extraindo bairro do endereço

In [67]:
import re

neighborhood_pattern = r'.* -\s*(.+)$'

df_chamados = df_chamados_raw.copy()
df_chamados['bairro'] = df_chamados['end'].astype(str).str.extract(neighborhood_pattern, flags=re.IGNORECASE)[0]
df_chamados['bairro'] = df_chamados['bairro'].str.strip()

Resolvendo problemas no registro de bairro de chamados

In [68]:
import unicodedata

# 1. Define abbreviation_map
abbreviation_map = {
    'JARDIM': 'JD',
    'VILA': 'VL',
    'PARQUE': 'PQ',
    'CONJUNTO RESIDENCIAL': 'CJ RES',
    'CIDADE': 'CD',
    'SETOR': 'ST',
    'DISTRITO INDUSTRIAL': 'DIST IND',
    'NUCLEO HABITACIONAL': 'NUC HAB',
    'RESIDENCIAL': 'RES'
}

# 2. Define specific_corrections_map
specific_corrections_map = {
    'VARZEA DO TAMANDUATE': 'VARZEA DO TAMANDUATEI',
    'VL FRANCISCO MATARAZ': 'VL FRANCISCO MATARAZZO',
    'PQ GERASSI CENTREVIL': 'PQ GERASSI',
    'JARDIM CLUBE DE CAMP': 'JD CLUBE DE CAMPO', # Correcting truncated name
    'ESTANCIA DO RIO GRAN': 'ESTANCIA DO RIO GRANDE', # Correcting truncated name
    'RECREIO DA BORDA DO': 'RECREIO DA BORDA DO CAMPO', # Correcting truncated name
    'ACAMPAMENTO ANCHIETA': 'ACAMPAMENTO ANCHIETA', # Keep as is, it's a specific area name
    'ASS. ESPIRITO SANTO, 117': 'JD ESPIRITO SANTO', # Assuming this refers to a neighborhood
    'BAIRRO INEXISTENTE': 'BAIRRO INEXISTENTE', # Keep as is, if it's truly an unknown bairro
    'CAMPO GRANDE': 'CAMPO GRANDE', # Keep as is
    'JARDIM DO MIRANTE': 'JD DO MIRANTE', # Standardizing
    'JARDIM SANTO ANDRÉ': 'JD SANTO ANDRE', # Standardizing
    'PARANAPIACABA': 'PARANAPIACABA', # Keep as is
    'RIO GRANDE': 'RIO GRANDE', # Keep as is
    'SITIO TAQUARAL': 'SITIO TAQUARAL', # Keep as is
    'TAMANDUATEÍ 2': 'TAMANDUATEI 2', # Standardizing accent
    'TAMANDUATEÍ 3': 'TAMANDUATEI 3', # Standardizing accent
    'TAMANDUATEÍ 8': 'TAMANDUATEI 8', # Standardizing accent
    'VARZEA DO TAMANDUATEI': 'VARZEA DO TAMANDUATEI', # Already corrected or correct
    'VILA JOÃO RAMALHO': 'VL JOAO RAMALHO', # Standardizing
    'VL FRANCISCO MATARAZZO': 'VL FRANCISCO MATARAZZO' # Already corrected or correct
}

# 3. Define function to remove accents and convert to uppercase
def remove_accents(text):
    if pd.isna(text):
        return text
    text = str(text).upper()
    nfkd_form = unicodedata.normalize('NFKD', text)
    only_ascii = nfkd_form.encode('ascii', 'ignore').decode('utf-8')
    return only_ascii.strip()

# 4. Apply remove_accents function
df_chamados['bairro'] = df_chamados['bairro'].apply(remove_accents)

# 5. Apply abbreviation_map
for full_form, abbr in abbreviation_map.items():
    df_chamados['bairro'] = df_chamados['bairro'].str.replace(full_form, abbr, regex=False)

# 6. Apply specific_corrections_map
for misspelled, corrected in specific_corrections_map.items():
    df_chamados['bairro'] = df_chamados['bairro'].str.replace(misspelled, corrected, regex=False)


print("Bairro standardization complete. Displaying unique values after transformation:")
print(df_chamados['bairro'].unique())

Bairro standardization complete. Displaying unique values after transformation:
['VL VITORIA' 'RECREIO DA BORDA DO CAMPO' 'VL ALZIRA' 'VL BASTOS'
 'VL METALURGICA' 'JD RIVIERA' 'VL PALMARES' 'JD SANTO ANDRE' 'VL LUZITA'
 'CENTRO' 'PQ DAS NACOES' 'JD SANTA CRISTINA' 'VL LINDA' 'SANTA MARIA'
 'VL GUIOMAR' 'VL FLORESTA' 'CONDOMINIO MARACANA' 'VL JOAO RAMALHO'
 'CATA PRETA' 'VL GUARANI' 'JD SANTO ANTONIO' 'VL SACADURA CABRAL'
 'VL HOMERO THON' 'VL CURUCA' 'PQ CAPUAVA' 'JD UTINGA' 'VL ASSUNCAO'
 'VL GILDA' 'JD IPANEMA' 'VL HUMAITA' 'VL AMERICA' 'SILVEIRA' 'JD VL RICA'
 'JD TELLES DE MENEZES' 'PQ JOAO RAMALHO' 'PARAISO' 'JD' 'CAMPESTRE'
 'VL MARINA' 'VL SCARPELLI' 'VL SUICA' 'CD SAO JORGE' 'JD CRISTIANE'
 'VL PRINCIPE DE GALES' 'SANTA TEREZINHA' 'JD BOM PASTOR' 'VL LUCINDA'
 'VL HELENA' 'VL GUARACIABA' 'PQ ERASMO ASSUNCAO' 'JD ALZIRA FRANCO'
 'BANGU' 'PQ ORATORIO' 'PQ NOVO ORATORIO' 'JD STELLA' 'JD MILENA'
 'JD BELA VISTA' 'VL VALPARAISO' 'VL CAMILOPOLIS' 'JD ITAPOAN' 'JD IRENE'
 'VL PROGRES

Checando bairros que existem nos chamados mas não pertencem a nenhuma bacia

In [69]:
# Get all unique neighborhoods from df_chamados
chamados_bairros = set(df_chamados['bairro'].dropna().unique())

# Get all unique neighborhoods from the bacias_bairros dictionary
bacias_known_bairros = set()
for bacia, bairros_list in bacias_bairros.items():
    bacias_known_bairros.update(bairros_list)

# Find bairros in df_chamados that are not in bacias_bairros
bairros_not_in_bacias = chamados_bairros - bacias_known_bairros

if bairros_not_in_bacias:
    print("Os seguintes bairros existem em 'df_chamados' mas não foram encontrados em nenhuma bacia do dicionário 'bacias_bairros':")
    for bairro in sorted(list(bairros_not_in_bacias)):
        print(f"- {bairro}")
else:
    print("Todos os bairros em 'df_chamados' foram encontrados em alguma bacia do dicionário 'bacias_bairros'.")

Os seguintes bairros existem em 'df_chamados' mas não foram encontrados em nenhuma bacia do dicionário 'bacias_bairros':
- ACAMPAMENTO ANCHIETA
- BAIRRO INEXISTENTE
- CAMPO GRANDE
- CD SAO JORGE
- ESTANCIA DO RIO GRANDE
- JD CLUBE DE CAMP
- JD CLUBE DE CAMPO
- JD DO MIRANTE
- JD ESPIRITO SANTO
- JD GUARIPOCABA
- JD JOAQUIM EUGENIO D
- JD RIVIERA
- JD SANTO ANTONIO DE
- JD SILVANA
- JD SILVIA
- PARANAPIACABA
- PQ AMERICA
- PQ DAS GARCAS
- PQ DO PEDROSO
- PQ JOAO RAMALHO
- PQ MIAMI
- PQ REPRESA BILLINGS
- PQ RIO GRANDE
- RECREIO DA BORDA DO CAMPO
- RIO GRANDE
- SITIO TAQUARAL


In [70]:
df_chamados_por_bacia = df_chamados.copy()

for bacia_name, bairros_list in bacias_bairros.items():
    df_chamados_por_bacia[bacia_name] = df_chamados['bairro'].isin(bairros_list).astype(int)

In [71]:
basin_columns = [col for col in df_chamados_por_bacia.columns if col in bacias_bairros.keys()]

if basin_columns:
    basin_proportions = df_chamados_por_bacia[basin_columns].sum() / len(df_chamados_por_bacia)
    print("Proporção de eventos por bacia (usando df_chamados_por_bacia):")
    print(basin_proportions.sort_values(ascending=False))
else:
    print("Nenhuma coluna de bacia encontrada no DataFrame df_chamados_por_bacia para calcular as proporções.")

Proporção de eventos por bacia (usando df_chamados_por_bacia):
tamanduatei    0.621775
guarara        0.329246
meninos        0.120822
oratorio       0.104104
dtype: float64


In [74]:
df_chamados_por_bacia.to_csv("chamados_por_bacia.csv", index=False)

In [75]:
df_chamados_por_bacia.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60353 entries, 0 to 60352
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   servico      60353 non-null  object
 1   end          60352 non-null  object
 2   data         60352 non-null  object
 3   enc          60159 non-null  object
 4   solicitante  60352 non-null  object
 5   bairro       60351 non-null  object
 6   meninos      60353 non-null  int64 
 7   oratorio     60353 non-null  int64 
 8   tamanduatei  60353 non-null  int64 
 9   guarara      60353 non-null  int64 
dtypes: int64(4), object(6)
memory usage: 4.6+ MB


# Removendo chamados que não são relacionados a chuvas

In [76]:
# Filtrando por tipo de serviço
enchente = ["809.3 - DDC - Enchente / Inundação / Alagamento - Núcleo",
            "809 - DDC - Enchente / Inundação / Alagamento",
            "809.2 - DDC - Enchente / Inundação / Alagamento - Residência",
            "809.1 - DDC - Enchente / Inundação / Alagamento - Via",
            "809.4 - DDC - Enchente / Inundação / Alagamento - Industria / Comércio"]

In [78]:
df_chamados_enchente = df_chamados_por_bacia[df_chamados_por_bacia['servico'].isin(enchente)].copy()

In [80]:
df_chamados_enchente.to_csv("chamados_enchente.csv", index=False)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [79]:
df_chamados_enchente.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2991 entries, 167 to 60165
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   servico      2991 non-null   object
 1   end          2991 non-null   object
 2   data         2991 non-null   object
 3   enc          2981 non-null   object
 4   solicitante  2991 non-null   object
 5   bairro       2991 non-null   object
 6   meninos      2991 non-null   int64 
 7   oratorio     2991 non-null   int64 
 8   tamanduatei  2991 non-null   int64 
 9   guarara      2991 non-null   int64 
dtypes: int64(4), object(6)
memory usage: 257.0+ KB


In [81]:
basin_columns = [col for col in df_chamados_enchente.columns if col in bacias_bairros.keys()]

if basin_columns:
    basin_proportions = df_chamados_enchente[basin_columns].sum() / len(df_chamados_enchente)
    print("Proporção de eventos por bacia (usando df_chamados_enchente):")
    print(basin_proportions.sort_values(ascending=False))
else:
    print("Nenhuma coluna de bacia encontrada no DataFrame df_chamados_enchente para calcular as proporções.")

Proporção de eventos por bacia (usando df_chamados_enchente):
tamanduatei    0.637245
guarara        0.392177
meninos        0.240388
oratorio       0.068205
dtype: float64


# Agregando por dia

Antes de mais nada, precisamos arrumar a coluna de data

In [87]:
df_chamados_enchente_dt = df_chamados_enchente.copy()
df_chamados_enchente_dt['data'] = pd.to_datetime(df_chamados_enchente_dt['data'], dayfirst=True)

In [89]:
basin_columns = ['tamanduatei', 'oratorio', 'meninos', 'guarara']
df_chamados_diario = df_chamados_enchente_dt.groupby('data')[basin_columns].sum().reset_index()

display(df_chamados_diario.head())

,data,tamanduatei,oratorio,meninos,guarara
0,2002-02-22,0,0,1,0
1,2004-09-15,10,0,0,10
2,2004-10-25,1,0,1,0
3,2004-12-06,4,0,2,20
4,2005-01-05,1,0,2,1


In [93]:
df_chamados_diario.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 474 entries, 0 to 473
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   data         474 non-null    datetime64[ns]
 1   tamanduatei  474 non-null    int64         
 2   oratorio     474 non-null    int64         
 3   meninos      474 non-null    int64         
 4   guarara      474 non-null    int64         
dtypes: datetime64[ns](1), int64(4)
memory usage: 18.6 KB


In [91]:
df_chamados_diario.to_csv("chamados_diario.csv", index=False)

In [92]:
days_with_rain_per_basin = {}
basin_columns = ['tamanduatei', 'oratorio', 'meninos', 'guarara']

for bacia in basin_columns:
    # Count days where the call count for the basin is greater than 0
    days_with_rain_per_basin[bacia] = (df_chamados_diario[bacia] > 0).sum()

print("Número de dias com chamados de enchente por bacia:")
for bacia, count in days_with_rain_per_basin.items():
    print(f"- {bacia.capitalize()}: {count} dias")

Número de dias com chamados de enchente por bacia:
- Tamanduatei: 348 dias
- Oratorio: 98 dias
- Meninos: 196 dias
- Guarara: 245 dias


# Batendo com a lista de alagamentos confirmados

In [97]:
df_alagamentos_confirmados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79 entries, 0 to 78
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   dt         79 non-null     object 
 1   ano        79 non-null     int64  
 2   mes        79 non-null     int64  
 3   dia        79 non-null     int64  
 4   date_diff  78 non-null     float64
dtypes: float64(1), int64(3), object(1)
memory usage: 3.2+ KB


Primeiro vamos deixar esse datetime batendo com o padrão do df_chamados_diario

In [99]:
df_alagamentos_dt = df_alagamentos_confirmados.copy()
df_alagamentos_dt['data'] = pd.to_datetime(df_alagamentos_dt['ano'].astype(str) + '-' +
                                          df_alagamentos_dt['mes'].astype(str) + '-' +
                                          df_alagamentos_dt['dia'].astype(str))

df_alagamentos_dt = df_alagamentos_dt.drop(columns=['ano', 'mes', 'dia'])

print("Coluna 'data' criada e formatada corretamente em df_alagamentos_dt. Colunas 'ano', 'mes', 'dia' removidas.")
display(df_alagamentos_dt.head())

Coluna 'data' criada e formatada corretamente em df_alagamentos_dt. Colunas 'ano', 'mes', 'dia' removidas.


,dt,date_diff,data
0,2016-01-10,NaN,2016-01-10
1,2016-01-15,4.0,2016-01-15
2,2016-02-05,20.0,2016-02-05
3,2016-02-15,10.0,2016-02-15
4,2016-02-24,8.0,2016-02-24


In [102]:
basin_columns = ['tamanduatei', 'oratorio', 'meninos', 'guarara'] # Define basin columns
df_chamados_confirmados = pd.merge(df_alagamentos_dt, df_chamados_diario, on='data', how='left')

# Fill NaN values in basin columns with 0 for dates present in df_alagamentos_dt but not in df_chamados_diario
df_chamados_confirmados[basin_columns] = df_chamados_confirmados[basin_columns].fillna(0).astype(int)

print("df_chamados_confirmados criado com sucesso a partir da mesclagem dos DataFrames, com valores zerados para datas sem correspondência.")
display(df_chamados_confirmados.head())

df_chamados_confirmados criado com sucesso a partir da mesclagem dos DataFrames, com valores zerados para datas sem correspondência.


,dt,date_diff,data,tamanduatei,oratorio,meninos,guarara
0,2016-01-10,NaN,2016-01-10,12,0,0,12
1,2016-01-15,4.0,2016-01-15,0,0,0,0
2,2016-02-05,20.0,2016-02-05,1,1,0,0
3,2016-02-15,10.0,2016-02-15,4,0,0,2
4,2016-02-24,8.0,2016-02-24,8,11,0,0


In [103]:
df_chamados_confirmados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79 entries, 0 to 78
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   dt           79 non-null     object        
 1   date_diff    78 non-null     float64       
 2   data         79 non-null     datetime64[ns]
 3   tamanduatei  79 non-null     int64         
 4   oratorio     79 non-null     int64         
 5   meninos      79 non-null     int64         
 6   guarara      79 non-null     int64         
dtypes: datetime64[ns](1), float64(1), int64(4), object(1)
memory usage: 4.4+ KB


In [105]:
basin_columns = ['tamanduatei', 'oratorio', 'meninos', 'guarara']

days_with_confirmation_no_calls = df_chamados_confirmados[
    df_chamados_confirmados[basin_columns].sum(axis=1) == 0
]

num_days = len(days_with_confirmation_no_calls)

print(f"Número de dias com confirmação de alagamento, mas nenhum chamado registrado nas bacias: {num_days}")

Número de dias com confirmação de alagamento, mas nenhum chamado registrado nas bacias: 39


In [110]:
days_with_calls_in_window = []

basin_columns = ['tamanduatei', 'oratorio', 'meninos', 'guarara']

for date in days_with_confirmation_no_calls['data']:
    start_date = date - pd.Timedelta(days=4)
    end_date = date + pd.Timedelta(days=4)

    # Filter df_chamados_diario for the 5-day window
    calls_in_window = df_chamados_diario[
        (df_chamados_diario['data'] >= start_date) &
        (df_chamados_diario['data'] <= end_date)
    ]

    # Check if there are any flood calls in any basin within this window
    if not calls_in_window.empty and calls_in_window[basin_columns].sum().sum() > 0:
        days_with_calls_in_window.append(date)

print(f"Original days with confirmation but no direct calls: {len(days_with_confirmation_no_calls)}")
print(f"Number of these days that now have associated flood calls within a 5-day window: {len(days_with_calls_in_window)}")

Original days with confirmation but no direct calls: 39
Number of these days that now have associated flood calls within a 5-day window: 6


**Reasoning**:
Filter the `df_chamados_diario` DataFrame to create `df_chamados_diario_filtered` containing entries where the 'data' column is on or after '2016-01-10', and then display relevant information about the new DataFrame.



In [116]:
df_chamados_diario_filtered = df_chamados_diario[df_chamados_diario['data'] >= '2016-01-10'].copy()

print(f"Number of entries in original df_chamados_diario: {len(df_chamados_diario)}")
print(f"Number of entries in df_chamados_diario_filtered: {len(df_chamados_diario_filtered)}")

display(df_chamados_diario_filtered.head())

Number of entries in original df_chamados_diario: 474
Number of entries in df_chamados_diario_filtered: 155


,data,tamanduatei,oratorio,meninos,guarara
319,2016-01-10,12,0,0,12
320,2016-01-11,2,1,0,1
321,2016-02-05,1,1,0,0
322,2016-02-15,4,0,0,2
323,2016-02-16,2,0,0,2


**Reasoning**:
To identify unconfirmed days, I need to find the dates present in `df_chamados_diario_filtered` that are not found in `df_alagamentos_dt` (which represents the confirmed flood events). This will create a list of dates that had flood calls but were not officially confirmed.



In [118]:
confirmed_dates = set(df_alagamentos_dt['data'].dt.date)

unconfirmed_flood_call_dates = df_chamados_diario_filtered[
    ~df_chamados_diario_filtered['data'].dt.date.isin(confirmed_dates)
].copy()

print(f"Number of daily flood calls that are not officially confirmed: {len(unconfirmed_flood_call_dates)}")
display(unconfirmed_flood_call_dates.head())

Number of daily flood calls that are not officially confirmed: 114


,data,tamanduatei,oratorio,meninos,guarara
320,2016-01-11,2,1,0,1
323,2016-02-16,2,0,0,2
325,2016-02-25,0,2,0,0
326,2016-06-06,3,0,1,1
327,2016-10-13,1,0,0,1


**Reasoning**:
To identify days with more than 5 flood calls in any basin among the unconfirmed dates, I will filter the `unconfirmed_flood_call_dates` DataFrame by checking if the count in any of the basin columns ('tamanduatei', 'oratorio', 'meninos', 'guarara') exceeds 5 for each day.



In [120]:
basin_columns = ['tamanduatei', 'oratorio', 'meninos', 'guarara']
high_call_unconfirmed_days = unconfirmed_flood_call_dates[
    (unconfirmed_flood_call_dates[basin_columns] > 5).any(axis=1)
].copy()

print(f"Number of unconfirmed days with more than 5 flood calls in any basin: {len(high_call_unconfirmed_days)}")
display(high_call_unconfirmed_days.head())

Number of unconfirmed days with more than 5 flood calls in any basin: 38


,data,tamanduatei,oratorio,meninos,guarara
343,2017-04-07,13,5,0,6
349,2017-11-21,7,0,0,6
361,2018-03-21,40,0,0,40
362,2018-03-22,73,0,0,75
372,2018-11-24,75,2,0,58


**Reasoning**:
To present the final result, I need to extract and list the specific dates from the `high_call_unconfirmed_days` DataFrame.



In [121]:
print("Dates with more than 5 flood calls in any basin, not officially confirmed, and on or after 2016-01-10:")
for date in high_call_unconfirmed_days['data']:
    print(date.strftime('%Y-%m-%d'))

Dates with more than 5 flood calls in any basin, not officially confirmed, and on or after 2016-01-10:
2017-04-07
2017-11-21
2018-03-21
2018-03-22
2018-11-24
2018-12-05
2019-03-11
2019-03-12
2019-03-14
2019-03-15
2019-03-16
2019-03-17
2019-03-18
2019-03-19
2019-03-20
2019-03-21
2019-03-22
2019-03-24
2019-03-26
2019-03-27
2019-03-29
2019-04-02
2019-04-04
2019-04-08
2019-04-09
2019-04-10
2019-04-11
2019-04-12
2019-04-15
2019-04-17
2019-04-18
2020-02-08
2020-02-09
2020-02-11
2020-02-19
2020-02-26
2020-03-02
2020-03-12


Summary:

Q&A
The number of days that had more than 5 flood calls in any basin but were not confirmed as flood events, considering only `df_chamados_diario` from 2016-01-10 onwards, is 38.

Data Analysis Key Findings
*   The initial daily flood call dataset (`df_chamados_diario`) contained 474 entries. After filtering for dates on or after 2016-01-10, the dataset was reduced to 155 entries.
*   Out of these 155 entries, 114 days were identified as having flood calls that were not officially confirmed as flood events.
*   From the unconfirmed days, 38 specific days had more than 5 flood calls in at least one basin.
*   These 38 days span from 2017 to 2020 and include multiple consecutive days in March and April 2019, such as '2019-03-11', '2019-03-12', '2019-03-14', '2019-03-15', '2019-03-16', '2019-03-17', '2019-03-18', '2019-03-19', '2019-03-20', '2019-03-21', '2019-03-22', '2019-03-24', '2019-03-26', '2019-03-27', '2019-03-29', '2019-04-02', '2019-04-04', '2019-04-08', '2019-04-09', '2019-04-10', '2019-04-11', '2019-04-12', '2019-04-15', '2019-04-17', and '2019-04-18'.

Insights or Next Steps
*   Investigate the reasons why these 38 days with a high volume of flood calls were not officially confirmed as flood events. This could reveal discrepancies in reporting, confirmation criteria, or the nature of the calls themselves (e.g., false alarms, minor incidents not meeting confirmation thresholds).
*   Analyze additional data sources for these specific unconfirmed days, such as weather patterns, news reports, or social media activity, to understand the context behind the high call volume.


# Validando por quantidade de chuva